In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.special import expit
from scipy import stats

import shap
import joblib

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report,
    precision_score, recall_score, f1_score, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.inspection import PartialDependenceDisplay

warnings.filterwarnings("ignore")

ARTIFACTS = Path("model_artifacts")
SEED = 42
rng = np.random.default_rng(SEED)

def find_optimal_threshold_f1(y_true, y_prob):
    """Umbral que maximiza F1 — mismo criterio usado en 2_models.ipynb."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = np.divide(
        2 * (precision * recall),
        (precision + recall),
        out=np.zeros_like(precision),
        where=(precision + recall) != 0,
    )
    best_idx = np.argmax(f1_scores)
    if best_idx == len(thresholds):
        best_idx -= 1
    return float(thresholds[best_idx])

# ── Cargar artefactos ─────────────────────────────────────────────────────────
m1  = joblib.load(ARTIFACTS / "m1_lr_baseline.pkl")
m2a = joblib.load(ARTIFACTS / "m2a_lr_era.pkl")

m3_post = np.load(ARTIFACTS / "m3_multilevel_advi.npz")

prep_common = joblib.load(ARTIFACTS / "preprocessor_common.pkl")
prep_m3     = joblib.load(ARTIFACTS / "preprocessor_m3.pkl")

arrs = np.load(ARTIFACTS / "test_arrays.npz")
X_test_sk        = arrs["X_test_sk"]    # ya preprocesado (58 características)
X_test_m3        = arrs["X_test_m3"]   # enriquecido con interacciones (62 características)
y_test           = arrs["y_test"]
era_test         = arrs["era_test"]
school_test      = arrs["school_test"]
X_train_bg       = arrs["X_train_sk_sample"]   # fondo para SHAP, ya preprocesado
y_train          = arrs["y_train"]
era_train        = arrs["era_train"]

with open(ARTIFACTS / "feature_names.json") as f:
    fnames = json.load(f)
feat_common  = fnames["common"]
feat_m3      = fnames["m3"]
era_labels   = fnames["era_labels"]
school_labels = fnames["school_labels"]

# X_test_sk / X_test_m3 ya están preprocesados — omitir el preprocesamiento del pipeline
# y llamar directamente al clasificador subyacente.
clf_m1  = m1.named_steps["clf"]
clf_m2a = m2a.named_steps["clf"]

# Pre-calcular todas las probabilidades
prob_m1  = clf_m1.predict_proba(X_test_sk)[:, 1]
# M2 usa el preprocesador enriquecido con interacciones (misma matriz que M3)
prob_m2a = clf_m2a.predict_proba(X_test_m3)[:, 1]
# beta_era es ahora un arreglo delta por escuela
prob_m3 = expit(
    m3_post["alpha_school"][school_test]
    + m3_post["beta_era"][school_test] * era_test
    + X_test_m3 @ m3_post["beta"]
)

model_probs = {
    "M1 – Baseline LR":                prob_m1,
    "M2 – LR + Tec21 Interactions":    prob_m2a,
    "M3 – Multilevel ADVI":            prob_m3,
}

print("Artefactos cargados. Conjunto de prueba:", X_test_sk.shape, "| tasa de deserción:", y_test.mean().round(4))


---
## 3.1  Evaluación en el Conjunto de Prueba con Intervalos de Confianza

Evaluamos todos los modelos en el conjunto de prueba retenido (20 % de los datos, estratificado por era × deserción).
Se reportan intervalos de confianza bootstrap (n = 1 000 remuestras) para ROC-AUC, PR-AUC y Brier score.
El umbral de clasificación se elige por modelo maximizando F1, consistente con la metodología de `2_models.ipynb`.


In [ ]:
def bootstrap_metrics(y_true, y_prob, n_boot=1000, seed=42):
    rng_b = np.random.default_rng(seed)
    aucs, aps, briers = [], [], []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng_b.integers(0, n, n)
        if y_true[idx].sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
        aps.append(average_precision_score(y_true[idx], y_prob[idx]))
        briers.append(brier_score_loss(y_true[idx], y_prob[idx]))
    def ci(arr):
        return np.percentile(arr, [2.5, 97.5])
    return {
        "roc_auc":  (roc_auc_score(y_true, y_prob), *ci(aucs)),
        "pr_auc":   (average_precision_score(y_true, y_prob), *ci(aps)),
        "brier":    (brier_score_loss(y_true, y_prob), *ci(briers)),
    }

rows = []
for name, prob in model_probs.items():
    thr = find_optimal_threshold_f1(y_test, prob)
    y_pred = (prob >= thr).astype(int)
    m = bootstrap_metrics(y_test, prob)
    rows.append({
        "Modelo": name,
        "ROC-AUC": f"{m['roc_auc'][0]:.4f} [{m['roc_auc'][1]:.4f}–{m['roc_auc'][2]:.4f}]",
        "PR-AUC":  f"{m['pr_auc'][0]:.4f} [{m['pr_auc'][1]:.4f}–{m['pr_auc'][2]:.4f}]",
        "Brier":   f"{m['brier'][0]:.4f} [{m['brier'][1]:.4f}–{m['brier'][2]:.4f}]",
        "Umbral (F1-ópt.)": f"{thr:.3f}",
        "F1":        f"{f1_score(y_test, y_pred, zero_division=0):.3f}",
        "Recall":    f"{recall_score(y_test, y_pred):.3f}",
        "Precisión": f"{precision_score(y_test, y_pred, zero_division=0):.3f}",
    })

ci_df = pd.DataFrame(rows)
display(ci_df.set_index("Modelo"))
print("\nNota: umbral elegido para maximizar F1 por modelo (consistente con 2_models.ipynb)")
ci_df.to_csv("ci_metrics.csv", index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, prob in model_probs.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, prob, name=name, ax=axes[1])

axes[0].set_title("Curvas ROC — Conjunto de Prueba")
axes[0].legend(fontsize=8)
axes[1].set_title("Curvas Precisión-Recall — Conjunto de Prueba\n(más informativas para datos desbalanceados)")
axes[1].legend(fontsize=8, loc="upper right")
fig.tight_layout()
plt.savefig("fig_roc_pr.png", dpi=150)
plt.show()


> **Preguntas de interpretación — Curvas ROC y PR**
>
> Responde las siguientes preguntas en tu reporte:
> 1. ¿Qué modelo alcanza el PR-AUC más alto? ¿Por cuántos puntos supera al modelo base?
>    ¿Esta diferencia es prácticamente significativa dado los intervalos de confianza?
> 2. La línea base de la curva PR (clasificador aleatorio) equivale a la prevalencia de la clase (~8 %). ¿Qué tan por encima
>    de esta línea base se ubican los mejores modelos a recall = 0.75?
> 3. Las curvas ROC se agrupan estrechamente (0.71–0.75 AUC). ¿Qué te dice esto sobre el valor marginal
>    de añadir estructura de era/escuela frente a las características crudas para discriminar desertores?
> 4. Compara ROC-AUC vs PR-AUC como criterios de evaluación para este caso (8 % de prevalencia).
>    ¿Cuál debería ser la métrica principal para un sistema de alerta temprana y por qué?


---
## 3.2  Análisis de Interpretabilidad

### 3.2.1  Regresión Logística — Coeficientes (M1 y M2)


In [ ]:
# ICs de coeficientes bootstrap para M1
n_boot = 500
coef_boot = []
rng_b = np.random.default_rng(SEED)
n = len(y_train)
# NOTA: reajustar 500 veces es lento; se usa el coef. almacenado como aproximación.
# Aquí se reportan estimaciones puntuales + ordenamiento por rango, destacando las 20 principales por |coef|.

coef_series = pd.Series(clf_m1.coef_[0], index=feat_common).sort_values(key=abs, ascending=False)
top_coefs = coef_series.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d62728" if v > 0 else "#1f77b4" for v in top_coefs.values]
ax.barh(top_coefs.index[::-1], top_coefs.values[::-1], color=colors[::-1])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Coeficiente (escala log-odds, características estandarizadas)")
ax.set_title("M1 – Top 20 Variables por |Coeficiente|\n(rojo = factor de riesgo, azul = factor protector)")
fig.tight_layout()
plt.savefig("fig_lr_coefs.png", dpi=150)
plt.show()

# Coeficiente de era en M2 — buscar la posición de era_main en la lista de características de interacción
era_feat_idx = list(feat_m3).index("era_main")
era_coef = clf_m2a.coef_[0][era_feat_idx]
print(f"\nM2 – coeficiente era_main: {era_coef:.4f}  (OR = {np.exp(era_coef):.3f})")
print("Tec21 está asociado con", "MAYOR" if era_coef > 0 else "MENOR", "log-odds de deserción (efecto principal, antes de interacciones)")


> ** Preguntas de interpretación — Coeficientes de Regresión Logística**
>
> 1. Los coeficientes positivos más altos (rojo) representan **factores de riesgo** para la deserción. Nombra los 3 principales
>    y explica su significado sustantivo en el contexto TEC. ¿Son consistentes con la teoría previa sobre retención estudiantil (p. ej., el modelo de Tinto)?
> 2. Los coeficientes negativos más altos (azul) representan **factores protectores**. Interpreta las
>    características relacionadas con becas (`scholarship.perc`, `scholarship.type`): ¿qué revelan sobre el papel del apoyo financiero?
> 3. El grupo de variables `activity_missing_flag` aparece con un coeficiente positivo grande. ¿Es esto
>    una señal de riesgo genuina o un artefacto de calidad de datos? ¿Qué recomendarías?
> 4. M2 añade `era_main` como efecto fijo. Interpreta el coeficiente: ¿la reforma Tec21 parece reducir el riesgo de deserción
>    después de controlar por covariables individuales? ¿Cuáles son las preocupaciones de identificación causal?


### 3.2.2  Valores SHAP — Explicaciones Globales y Locales (M1)


In [ ]:
# LinearExplainer necesita el estimador LR directo — X_train_bg ya está preprocesado
explainer_m1 = shap.LinearExplainer(clf_m1, X_train_bg, feature_names=feat_common)
shap_vals_m1 = explainer_m1(X_test_sk)

# ── SHAP Beeswarm ────────────────────────────────────────────────────────────
# NOTA: No pre-crear fig/ax — SHAP beeswarm gestiona su propia figura.
# Llamar plt.subplots() antes de shap.plots.beeswarm() hace que la gráfica
# se renderice en ejes vacíos (conflicto conocido entre shap y matplotlib).
plt.close("all")
shap.plots.beeswarm(shap_vals_m1, max_display=20, show=False)
plt.title("SHAP Beeswarm — M1 Regresión Logística Base (importancia global de variables)")
plt.tight_layout()
plt.savefig("fig_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


In [ ]:
plt.close("all")
shap.plots.bar(shap_vals_m1, max_display=15, show=False)
plt.title("SHAP Media |valor| — Importancia global de variables")
plt.tight_layout()
plt.savefig("fig_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

top_idx = int(np.argmax(prob_m1))
print(f"Explicando observación {top_idx}: P(deserción) predicha = {prob_m1[top_idx]:.3f}")
plt.close("all")
shap.plots.waterfall(shap_vals_m1[top_idx], max_display=12, show=False)
plt.title(f"SHAP Waterfall — estudiante de mayor riesgo (idx={top_idx})")
plt.tight_layout()
plt.savefig("fig_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


> **Preguntas de interpretación — Valores SHAP**
>
> 1. **Beeswarm**: El eje horizontal es el valor SHAP (impacto en log-odds). Identifica la variable
>    con mayor dispersión. ¿Qué indica una dispersión amplia sobre la heterogeneidad de su efecto entre estudiantes?
> 2. ¿Los rankings de SHAP coinciden con los rankings de coeficientes crudos de la celda anterior? Si difieren,
>    explica por qué SHAP puede dar una imagen diferente incluso para un modelo lineal (pista: correlación entre variables y efectos de tasa base).
> 3. **Waterfall (estudiante individual)**: Analiza las 3 variables más influyentes que empujan
>    a este estudiante hacia la deserción. ¿Son variables sobre las que la institución puede actuar
>    (accionables) o características fijas (no accionables)?
> 4. Conecta estos hallazgos con tu pregunta de investigación: **¿qué has aprendido sobre los
>    mecanismos de deserción** en el TEC? ¿Las principales variables son de naturaleza académica, socioeconómica o institucional?


### 3.2.3  Gráficas de Dependencia Parcial — Variables Principales


In [ ]:
# Identificar las 5 variables numéricas principales por SHAP medio |valor|
mean_abs_shap = np.abs(shap_vals_m1.values).mean(axis=0)
top5_idx = np.argsort(mean_abs_shap)[::-1][:5]
top5_names = [feat_common[i] for i in top5_idx]
print("Top 5 variables para PDP:", top5_names)

X_eval = X_test_sk.astype("float64")   # sklearn internamente requiere float64

fig, axes = plt.subplots(1, len(top5_idx), figsize=(4 * len(top5_idx), 4), sharey=False)

for ax, feat_i, name in zip(axes, top5_idx.tolist(), top5_names):
    col = X_eval[:, feat_i]
    uniq = np.unique(col)
    grid = uniq if len(uniq) <= 10 else np.linspace(np.percentile(col, 5), np.percentile(col, 95), 50)
    avg_preds = [
        clf_m1.predict_proba(np.where(np.arange(X_eval.shape[1]) == feat_i, v, X_eval))[:, 1].mean()
        for v in grid
    ]
    marker = "o" if len(grid) <= 10 else None
    ax.plot(grid, avg_preds, color="#1f77b4", marker=marker, ms=4)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel(name, fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("Gráficas de Dependencia Parcial — M1 Regresión Logística Base", y=1.02)
fig.tight_layout()
plt.savefig("fig_pdp.png", dpi=150, bbox_inches="tight")
plt.show()


> **📝 Preguntas de interpretación — Gráficas de Dependencia Parcial**
>
> 1. Para la variable numérica más importante, describe la forma de la curva PDP:
>    ¿el efecto es monótono, no lineal o plano? ¿Qué explicación sustantiva ofrecerías?
> 2. Las PDP muestran el efecto **marginal** promediando sobre todas las demás variables. ¿Qué supuesto
>    requiere esto, y cuándo podría violarse en tu conjunto de datos
>    (pista: piensa en variables específicas de la era)?
> 3. Compara la dirección de la pendiente de la PDP con el signo del coeficiente en la celda anterior.
>    ¿Están de acuerdo? (Deberían para un modelo lineal — confirma y explica.)
> 4. Si asesoraras a la oficina de retención del TEC, ¿cuál de las 5 principales variables destacarías
>    como la señal de alerta temprana más **accionable** y por qué?


---
## (IGNORAR) 3.2.4  Diagnóstico de Convergencia ADVI (M3 — Modelo Multinivel) (Deprecado)

ADVI (Inferencia Variacional por Diferenciación Automática) optimiza una cota inferior (ELBO) sobre la
log-verosimilitud marginal. Si el ELBO no ha alcanzado una meseta al final de las iteraciones, la aproximación
variacional no ha convergido y las medias posteriores no son confiables.

Se grafica la traza completa del ELBO y se calcula la mejora relativa en el último 20 % de las iteraciones
para cuantificar la convergencia.


In [ ]:
# ── Cargar historial del ELBO ─────────────────────────────────────────────────
elbo_path = ARTIFACTS / "advi_elbo_history_m3.npy"

if elbo_path.exists():
    elbo_hist = np.load(elbo_path)
    n_iter = len(elbo_hist)

    # Suavizar con ventana deslizante para mayor legibilidad
    window = max(1, n_iter // 100)
    elbo_smooth = pd.Series(elbo_hist).rolling(window, min_periods=1).mean().values

    # Métrica de convergencia: cambio relativo del ELBO en el último 20 % de las iteraciones
    cutoff = int(0.80 * n_iter)
    elbo_early  = elbo_smooth[cutoff]
    elbo_final  = elbo_smooth[-1]
    rel_change  = abs((elbo_final - elbo_early) / (abs(elbo_early) + 1e-8))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # ── Traza completa ───────────────────────────────────────────────────────
    axes[0].plot(elbo_hist,   color="lightgrey", lw=0.4, label="cruda")
    axes[0].plot(elbo_smooth, color="#1f77b4",   lw=1.5, label=f"suavizada (ventana={window})")
    axes[0].axvline(cutoff, color="red", ls="--", lw=0.8,
                    label=f"marca 80 % (iter {cutoff:,})")
    axes[0].set_xlabel("Iteración ADVI")
    axes[0].set_ylabel("ELBO negativo (pérdida; menor = mejor)")
    axes[0].set_title("M3 — Traza Completa de Pérdida ADVI")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    # ── Zoom último 20 % ─────────────────────────────────────────────────────
    axes[1].plot(np.arange(cutoff, n_iter), elbo_hist[cutoff:],
                 color="lightgrey", lw=0.4, label="cruda")
    axes[1].plot(np.arange(cutoff, n_iter), elbo_smooth[cutoff:],
                 color="#d62728", lw=1.5, label="suavizada")
    axes[1].set_xlabel("Iteración ADVI")
    axes[1].set_title(f"Último 20 % de iteraciones (zoom)\nCambio relativo: {rel_change:.4%}")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    plt.savefig("fig_advi_elbo.png", dpi=150)
    plt.show()

    # ── Resumen de convergencia ──────────────────────────────────────────────
    print(f"\n── Resumen de Convergencia ADVI ─────────────────────────────────────")
    print(f"  Total de iteraciones     : {n_iter:,}")
    print(f"  ELBO en iter 1           : {elbo_hist[0]:.1f}")
    if n_iter >= 5000:
        print(f"  ELBO en iter 5,000       : {elbo_hist[4999]:.1f}  (punto de parada original)")
        print(f"  Cambio ELBO 1→5k         : {elbo_hist[4999] - elbo_hist[0]:+.1f}")
    print(f"  ELBO en iter {n_iter:,}   : {elbo_hist[-1]:.1f}")
    if n_iter >= 5000:
        print(f"  Cambio ELBO 5k→{n_iter//1000}k      : {elbo_hist[-1] - elbo_hist[4999]:+.1f}")
    print(f"  Cambio rel. (último 20%) : {rel_change:.4%}")
    print()
    if rel_change < 0.001:
        print("  ✓ CONVERGIÓ  — cambio relativo < 0.1 % en el último 20 % de las iteraciones")
    elif rel_change < 0.01:
        print("  ⚠ PROBABLEMENTE CONVERGIÓ  — cambio relativo < 1 %; inspecciona la gráfica de zoom")
    else:
        print("  ✗ NO CONVERGIÓ  — cambio relativo > 1 %; considera más iteraciones")
        print("    → Si hay una meseta visible pero con pérdida alta, el ADVI de campo medio puede ser")
        print("      demasiado restrictivo; considera NUTS completo en una submuestra.")

else:
    print("⚠ Archivo de historial del ELBO no encontrado.")
    print("  Vuelve a ejecutar la celda de exportación en experiments.ipynb con el código actualizado del tracker.")
    print(f"  Ruta esperada: {elbo_path}")


> **Preguntas de interpretación — Convergencia ADVI**
>
> 1. **Traza completa**: ¿La curva de pérdida muestra un codo claro (caída rápida inicial, luego meseta)?
>    ¿Aproximadamente en qué iteración se aplana la curva? ¿Qué te dice esto sobre el número mínimo de iteraciones necesarias para este modelo?
>
> 2. **Zoom (último 20 %)**: ¿La curva suavizada sigue descendiendo, está aproximadamente plana o es ruidosa
>    alrededor de una meseta?
>    - Sigue descendiendo → no convergió; reporta los resultados de M3 con una advertencia
>    - Plana pero ruidosa → probablemente convergió; el ruido es varianza de Monte Carlo en el estimador del ELBO
>    - Meseta suave → convergió; los resultados son confiables
>
> 3. **Comparación 5k vs 10k**: ¿En cuántas unidades de ELBO mejoró la pérdida entre la iteración
>    5,000 y 10,000? ¿Es esta mejora significativa en relación con la caída total desde
>    la iteración 1 hasta 5,000?
>
> 4. **Implicación para el rendimiento de M3**: M3 tiene un ROC-AUC menor que M1 (base) y un
>    umbral implausiblemente bajo (0.007). Si el ELBO ha convergido, esto sugiere que la
>    **aproximación ADVI de campo medio en sí misma** es el cuello de botella — no el número de iteraciones.
>    El ADVI de campo medio asume que todos los parámetros son independientes, lo cual se viola en un
>    modelo multinivel donde los interceptos de escuela correlacionan con la media global.
>    Si el ELBO *no* ha convergido, el umbral bajo es simplemente un artefacto de una solución degenerada. Distingue estos dos casos basándote en las gráficas.
>
> 5. **Recomendación**: Con base en lo que observas, ¿recomendarías (a) ejecutar más iteraciones de ADVI,
>    (b) cambiar a NUTS completo en una submuestra estratificada, o (c) eliminar M3
>    de la comparación final? Justifica tu elección.


---
## 3.3  Análisis de Robustez

Se evalúa si las conclusiones son estables bajo diferentes opciones metodológicas.
Un resultado se considera **robusto** si el cambio en ROC-AUC es < 0.01 y en PR-AUC < 0.02.
El umbral de clasificación se elige maximizando F1 en cada modelo.


In [ ]:
thresholds_grid = np.linspace(0.05, 0.70, 60)

# 3 modelos, grilla 1×3
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.ravel()

for ax, (name, prob) in zip(axes, model_probs.items()):
    precs, recs, f1s = [], [], []
    for t in thresholds_grid:
        y_pred_t = (prob >= t).astype(int)
        precs.append(precision_score(y_test, y_pred_t, zero_division=0))
        recs.append(recall_score(y_test, y_pred_t))
        f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    thr_f1 = find_optimal_threshold_f1(y_test, prob)
    ax.plot(thresholds_grid, recs,   label="Recall",    color="#1f77b4")
    ax.plot(thresholds_grid, precs,  label="Precisión", color="#d62728")
    ax.plot(thresholds_grid, f1s,    label="F1",        color="#2ca02c", lw=2)
    ax.axvline(thr_f1, ls="--", color="grey", lw=0.9,
               label=f"umbral F1-ópt. = {thr_f1:.3f}")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Umbral de clasificación")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

fig.suptitle("Precisión / Recall / F1 vs Umbral — todos los modelos")
fig.tight_layout()
plt.savefig("fig_threshold_sensitivity.png", dpi=150)
plt.show()


In [ ]:
mask_pre  = era_test == 0
mask_tec  = era_test == 1

print(f"Pre-Tec21 (prueba): {mask_pre.sum():,} filas | deserción {y_test[mask_pre].mean():.3%}")
print(f"Tec21    (prueba):  {mask_tec.sum():,} filas | deserción {y_test[mask_tec].mean():.3%}\n")

sub_rows = []
for name, prob in model_probs.items():
    thr = find_optimal_threshold_f1(y_test, prob)
    for era_name, mask in [("Pre-Tec21", mask_pre), ("Tec21", mask_tec)]:
        sub_rows.append({
            "Modelo": name, "Era": era_name,
            "ROC-AUC":   roc_auc_score(y_test[mask], prob[mask]),
            "PR-AUC":    average_precision_score(y_test[mask], prob[mask]),
            "F1":        f1_score(y_test[mask], (prob[mask] >= thr).astype(int), zero_division=0),
            "Recall":    recall_score(y_test[mask], (prob[mask] >= thr).astype(int)),
            "Precisión": precision_score(y_test[mask], (prob[mask] >= thr).astype(int), zero_division=0),
        })

sub_df = pd.DataFrame(sub_rows)
display(sub_df.pivot(index="Modelo", columns="Era", values=["ROC-AUC","PR-AUC","F1","Recall","Precisión"])
        .round(4))


In [ ]:
school_rows = []
for school_id, school_name in enumerate(school_labels):
    mask_s = school_test == school_id
    if mask_s.sum() < 30 or y_test[mask_s].sum() < 5:
        continue
    for name, prob in model_probs.items():
        thr = find_optimal_threshold_f1(y_test, prob)
        school_rows.append({
            "Modelo": name, "Escuela": school_name,
            "n": int(mask_s.sum()),
            "tasa_desercion": float(y_test[mask_s].mean()),
            "ROC-AUC": roc_auc_score(y_test[mask_s], prob[mask_s]),
            "PR-AUC":  average_precision_score(y_test[mask_s], prob[mask_s]),
            "F1":      f1_score(y_test[mask_s], (prob[mask_s] >= thr).astype(int), zero_division=0),
            "Recall":  recall_score(y_test[mask_s], (prob[mask_s] >= thr).astype(int)),
        })

school_df = pd.DataFrame(school_rows)
display(school_df[school_df["Modelo"] == "M1 – Baseline LR"]
        .set_index("Escuela")[["n","tasa_desercion","ROC-AUC","PR-AUC","F1","Recall"]].round(4))


In [ ]:
# Construir la tabla de sensibilidad requerida por component3.md
base_m1_auc = roc_auc_score(y_test, prob_m1)
base_m1_ap  = average_precision_score(y_test, prob_m1)
base_thr    = find_optimal_threshold_f1(y_test, prob_m1)

sens_rows = [
    {
        "Variación": "Modelo base (M1)",
        "Métrica Base": f"AUC={base_m1_auc:.4f}",
        "Métrica Comparada": "—",
        "Δ AUC": "—",
        "Interpretación": "Punto de referencia"
    },
    {
        "Variación": "+ efecto fijo de era / interacciones (M2 vs M1)",
        "Métrica Base": f"AUC={base_m1_auc:.4f}",
        "Métrica Comparada": f"AUC={roc_auc_score(y_test, prob_m2a):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test, prob_m2a) - base_m1_auc:+.4f}",
        "Interpretación": "La agrupación por era añade discriminación casi nula"
    },
    {
        "Variación": "+ términos de interacción Tec21 (M2 vs M1)",
        "Métrica Base": f"AUC={base_m1_auc:.4f}",
        "Métrica Comparada": f"AUC={roc_auc_score(y_test, prob_m2a):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test, prob_m2a) - base_m1_auc:+.4f}",
        "Interpretación": "Las interacciones explícitas era×variable añaden discriminación marginal sobre el modelo base"
    },
    {
        "Variación": "Efectos aleatorios por escuela+era ADVI (M3 vs M1)",
        "Métrica Base": f"AUC={base_m1_auc:.4f}",
        "Métrica Comparada": f"AUC={roc_auc_score(y_test, prob_m3):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test, prob_m3) - base_m1_auc:+.4f}",
        "Interpretación": "Los efectos aleatorios por escuela reducen el AUC — el ADVI puede estar subajustado en 5k iteraciones"
    },
    {
        "Variación": "Umbral 0.5 → óptimo F1 (M1)",
        "Métrica Base": f"F1@0.5={f1_score(y_test,(prob_m1>=0.5).astype(int),zero_division=0):.3f}",
        "Métrica Comparada": f"F1@ópt={f1_score(y_test,(prob_m1>=base_thr).astype(int),zero_division=0):.3f}",
        "Δ AUC": "N/A (umbral)",
        "Interpretación": "El umbral óptimo F1 mejora el balance precisión-recall respecto al umbral por defecto (0.5)"
    },
    {
        "Variación": "Subgrupo por era: Pre-Tec21 vs Tec21 (M1)",
        "Métrica Base": f"AUC(Pre)={roc_auc_score(y_test[mask_pre], prob_m1[mask_pre]):.4f}",
        "Métrica Comparada": f"AUC(Tec)={roc_auc_score(y_test[mask_tec], prob_m1[mask_tec]):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test[mask_tec],prob_m1[mask_tec])-roc_auc_score(y_test[mask_pre],prob_m1[mask_pre]):+.4f}",
        "Interpretación": (
        f"Brecha significativa: el AUC cae {roc_auc_score(y_test[mask_tec], prob_m1[mask_tec]) - roc_auc_score(y_test[mask_pre], prob_m1[mask_pre]):.4f} "
        f"en Tec21 vs Pre-Tec21. El modelo entrenado principalmente en datos Pre-Tec21 ({mask_pre.sum():,} filas) "
        f"puede no capturar los patrones de deserción específicos de Tec21. La brecha supera el umbral de robustez (0.01)."
    )
    },
]

sens_df = pd.DataFrame(sens_rows)
display(sens_df.set_index("Variación"))
sens_df.to_csv("sensitivity_table.csv", index=False)


> **Preguntas de interpretación — Robustez**
>
> 1. **Sensibilidad al umbral**: El umbral F1-óptimo equilibra precisión y recall automáticamente.
>    ¿Cuántas revisiones por verdadero positivo resultan en ese punto? En un contexto de intervención TEC
>    (p. ej., contacto con asesores), ¿es ese balance aceptable? ¿En qué dirección moverías el umbral
>    (más recall, menos precisión vs. menos carga, mayor precisión) y por qué?
> 2. **Subgrupo por era**: Si el modelo se desempeña materialmente diferente en estudiantes Pre-Tec21 vs Tec21,
>    ¿cuáles son las implicaciones para desplegarlo en la era actual (Tec21)?
>    ¿Deberían entrenarse modelos separados por era?
> 3. **Subgrupo por escuela**: Identifica la escuela con la mayor brecha entre su tasa de deserción
>    y el AUC por escuela del modelo. ¿Qué podría explicar esto? (Considera tamaño de muestra,
>    factores no observados específicos de la escuela, calidad de los datos.)
> 4. **Veredicto general de robustez**: Resume en 2–3 oraciones si las conclusiones
>    (qué variables importan, qué modelo es mejor) son estables a través de las variaciones evaluadas.
>    Sé honesto sobre qué decisiones *sí* cambian materialmente los resultados.


---
## 3.4  Análisis de Errores — Falsos Positivos y Falsos Negativos

Se inspeccionan los patrones sistemáticos entre los estudiantes que el mejor modelo sklearn disponible (M1) clasifica incorrectamente.


In [ ]:
# Usar M1 con el umbral óptimo F1
thr_m1 = find_optimal_threshold_f1(y_test, prob_m1)
y_pred_m1 = (prob_m1 >= thr_m1).astype(int)

# Reconstruir la matriz de características de prueba original no está disponible desde el preprocesador,
# así que se trabaja con la matriz transformada + nombres de características.
test_df = pd.DataFrame(X_test_sk, columns=feat_common)
test_df["y_true"]  = y_test
test_df["y_pred"]  = y_pred_m1
test_df["prob"]    = prob_m1
test_df["era"]     = era_test
test_df["school"]  = school_test

FP = test_df[(test_df["y_pred"] == 1) & (test_df["y_true"] == 0)]
FN = test_df[(test_df["y_pred"] == 0) & (test_df["y_true"] == 1)]
TP = test_df[(test_df["y_pred"] == 1) & (test_df["y_true"] == 1)]
TN = test_df[(test_df["y_pred"] == 0) & (test_df["y_true"] == 0)]

print(f"Umbral F1-óptimo: {thr_m1:.3f}")
print(f"VP: {len(TP):,}  |  FP: {len(FP):,}  |  FN: {len(FN):,}  |  VN: {len(TN):,}")
print(f"F1: {f1_score(y_test, y_pred_m1, zero_division=0):.3f}  |  "
      f"Recall: {recall_score(y_test, y_pred_m1):.3f}  |  "
      f"Precisión: {precision_score(y_test, y_pred_m1, zero_division=0):.3f}")

# Comparar medias de variables numéricas entre grupos de error
numeric_feats = feat_common[:10]   # las primeras 10 son las numéricas (orden del StandardScaler)
compare_df = pd.DataFrame({
    "VP (desertor real, detectado)":    TP[numeric_feats].mean(),
    "FN (desertor real, no detectado)": FN[numeric_feats].mean(),
    "FP (retenido, marcado)":           FP[numeric_feats].mean(),
    "VN (retenido, correcto)":          TN[numeric_feats].mean(),
}).T.round(3)
display(compare_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de errores por era
era_map = {0: "Pre-Tec21", 1: "Tec21"}
for ax, (group_name, group_df) in zip(axes, [("Falsos Positivos", FP), ("Falsos Negativos", FN)]):
    era_counts = group_df["era"].map(era_map).value_counts()
    ax.bar(era_counts.index, era_counts.values, color=["#1f77b4","#ff7f0e"])
    ax.set_title(f"{group_name} por Era\n(n={len(group_df):,})")
    ax.set_ylabel("Cantidad")

fig.tight_layout()
plt.savefig("fig_fp_fn_era.png", dpi=150)
plt.show()

# Distribución de probabilidad de FP vs VP (¿qué tan seguro está el modelo en sus errores?)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(FP["prob"], bins=40, alpha=0.6, label=f"Falsos Positivos (n={len(FP):,})", color="#d62728")
ax.hist(TP["prob"], bins=40, alpha=0.6, label=f"Verdaderos Positivos  (n={len(TP):,})", color="#2ca02c")
ax.axvline(thr_m1, color="black", ls="--", label=f"umbral={thr_m1:.3f}")
ax.set_xlabel("P(deserción) predicha")
ax.set_title("Distribuciones de puntuación: VP vs FP")
ax.legend()
fig.tight_layout()
plt.savefig("fig_fp_score_dist.png", dpi=150)
plt.show()


In [ ]:
# ── Análisis de costo operacional: umbral vs carga de trabajo ──────────────────
# Barrido de umbrales para visualizar el compromiso F1 / recall / carga de trabajo.

review_rows = []
for t_val in np.arange(0.05, 0.70, 0.02):
    y_pred_t = (prob_m1 >= t_val).astype(int)
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    flagged = tp + fp
    if flagged == 0:
        continue
    review_rows.append({
        "Umbral":                round(t_val, 2),
        "Estudiantes marcados":  int(flagged),
        "Desertores detectados": int(tp),
        "Falsas alarmas":        int(fp),
        "Revisiones por VP":     round(flagged / max(tp, 1), 1),
        "Recall":                round(tp / (tp + fn + 1e-9), 3),
        "F1":                    round(f1_score(y_test, y_pred_t, zero_division=0), 3),
    })

cost_df = pd.DataFrame(review_rows)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.plot(cost_df["Umbral"], cost_df["Recall"],
         color="#1f77b4", lw=2, label="Recall")
ax1.plot(cost_df["Umbral"], cost_df["F1"],
         color="#2ca02c", lw=2, label="F1")
ax1.set_ylabel("Recall / F1", color="black")
ax1.set_ylim(0, 1)

ax2.plot(cost_df["Umbral"], cost_df["Revisiones por VP"],
         color="#d62728", lw=2, ls="--", label="Revisiones por verdadero positivo")
ax2.set_ylabel("Revisiones por verdadero positivo (carga de trabajo)", color="#d62728")

f1_thr = find_optimal_threshold_f1(y_test, prob_m1)
ax1.axvline(f1_thr, color="grey", ls=":", lw=1.2,
            label=f"umbral F1-óptimo = {f1_thr:.3f}")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="center right")

ax1.set_xlabel("Umbral de clasificación")
ax1.set_title("F1 / Recall vs Carga de Trabajo del Asesor (M1)\n"
              "Eje izquierdo: F1 y recall | Eje derecho: # estudiantes revisados por desertor real encontrado")
ax1.grid(alpha=0.3)
fig.tight_layout()
plt.savefig("fig_operational_cost.png", dpi=150)
plt.show()

# Imprimir la fila correspondiente al umbral F1-óptimo
closest = cost_df.iloc[(cost_df["Umbral"] - f1_thr).abs().argsort()[:1]]
print(f"\nEn umbral F1-óptimo ≈ {f1_thr:.3f}:")
display(closest)


> **Preguntas de interpretación — Análisis de Errores**
>
> **Falsos Positivos (estudiantes marcados como riesgo de deserción que permanecieron):**
> 1. En la tabla de comparación de variables numéricas, ¿qué variables son *más altas* para los estudiantes
>    FP comparados con los VN? ¿Qué tipo de estudiante es falsamente marcado — existe un perfil
>    (p. ej., bajo ingreso pero resiliente, o estudiantes con datos faltantes)?
> 2. ¿Los falsos positivos están desproporcionadamente concentrados en una era o escuela?
>    ¿Qué implica esto para la equidad en un sistema automatizado de alertas?
>
> **Falsos Negativos (estudiantes que desertaron pero no fueron detectados):**
> 3. ¿Qué perfil caracteriza al grupo FN comparado con los estudiantes VP?
>    ¿Son desertores "silenciosos" — estudiantes que parecen académicamente bien pero que se van por
>    razones no académicas (financieras, personales)?
> 4. El modelo no detecta ~25 % de las deserciones reales (objetivo de recall era 0.75). Desde una
>    perspectiva institucional, ¿estos errores son aleatorios o sistemáticos? Si son sistemáticos,
>    ¿qué datos adicionales solicitarías para reducirlos?
>
> **En general:**
> 5. ¿Estos patrones de error revelan alguna **preocupación de equidad** (desventaja sistemática
>    para algún subgrupo)? Referencia escuelas o eras específicas si los datos lo respaldan.
> 6. Resume en un párrafo: ¿cuáles son las **principales limitaciones** del enfoque de modelado actual,
>    y qué mejoras metodológicas priorizarías para la siguiente iteración?


---
## 3.5  Respuesta a la Pregunta de Investigación

> **¿De qué manera la transición al modelo Tec21 alteró la heterogeneidad estructural de la deserción entre escuelas, y qué variables han ganado peso predictivo en este nuevo régimen?**

Las dos subsecciones siguientes responden directamente esta pregunta:

- **§3.5.1** examina qué variables cambiaron de peso predictivo al pasar de M1 (sin era) a M2 (con términos de interacción Tec21).  
- **§3.5.2** cuantifica cómo varían entre escuelas el riesgo base de deserción y el impacto diferencial de la transición Tec21, usando los efectos aleatorios de M3.

### 3.5.1  Variables con Mayor Cambio de Peso: M1 vs M2 (Coeficientes de Interacción Tec21)

In [ ]:
feat_m3_list    = list(feat_m3)
feat_common_list = list(feat_common)

INTERACTION_TERMS = [
    'PNA_x_era',
    'scholarship.perc_x_era',
    'activity_count_unified_x_era',
    'admission.test_x_era',
]
BASE_VARS = ['PNA', 'scholarship.perc', 'activity_count_unified', 'admission.test']
SPANISH_LABELS = {
    'PNA':                      'Promedio Nal. Admisión',
    'scholarship.perc':         'Beca (%)',
    'activity_count_unified':   'Actividades extracurriculares',
    'admission.test':           'Prueba de admisión',
    'era_main':                 'Tec21 (efecto principal)',
}

# Coeficientes M1 (features comunes) y M2 (features m3)
coefs_m1 = clf_m1.coef_[0]   # len 58
coefs_m2 = clf_m2a.coef_[0]  # len 62

# ── Panel izquierdo: comparación de efectos principales M1 vs M2 ─────────────
m1_vals, m2_main_vals, labels_main = [], [], []
for bv in BASE_VARS:
    if bv in feat_common_list and bv in feat_m3_list:
        m1_vals.append(coefs_m1[feat_common_list.index(bv)])
        m2_main_vals.append(coefs_m2[feat_m3_list.index(bv)])
        labels_main.append(SPANISH_LABELS.get(bv, bv))

# Efecto principal era en M2
era_idx  = feat_m3_list.index('era_main')
era_coef_m2 = coefs_m2[era_idx]

# ── Panel derecho: coeficientes de interacción era×variable en M2 ─────────────
inter_vals, labels_inter = [], []
for it, bv in zip(INTERACTION_TERMS, BASE_VARS):
    if it in feat_m3_list:
        inter_vals.append(coefs_m2[feat_m3_list.index(it)])
        labels_inter.append(SPANISH_LABELS.get(bv, bv))

# ── Figura ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("M2: Impacto de los Términos de Interacción Tec21 en los Coeficientes", fontsize=13, fontweight='bold')

x = np.arange(len(labels_main))
w = 0.35
ax = axes[0]
bars_m1 = ax.bar(x - w/2, m1_vals,  w, label='M1 (sin era)',  color='steelblue',   alpha=0.8)
bars_m2 = ax.bar(x + w/2, m2_main_vals, w, label='M2 (efecto principal)', color='darkorange', alpha=0.8)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels(labels_main, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Coeficiente (log-odds)'); ax.set_title('Efectos Principales: M1 vs M2')
ax.legend(fontsize=9)

# Anotar efecto era principal
ax.annotate(f'era_main = {era_coef_m2:+.3f}',
            xy=(0.98, 0.96), xycoords='axes fraction',
            ha='right', va='top', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='grey', alpha=0.8))

ax2 = axes[1]
colors_inter = ['#d62728' if v > 0 else '#1f77b4' for v in inter_vals]
bars_i = ax2.bar(labels_inter, inter_vals, color=colors_inter, alpha=0.85)
ax2.axhline(0, color='black', lw=0.8, ls='--')
ax2.set_ylabel('Coeficiente de Interacción (log-odds)')
ax2.set_title('Términos de Interacción era×Variable en M2')
ax2.set_xticklabels(labels_inter, rotation=20, ha='right', fontsize=9)

for bar, val in zip(bars_i, inter_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + (0.003 if val >= 0 else -0.006),
             f'{val:+.3f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=8.5, fontweight='bold')

fig.tight_layout()
plt.savefig("../outputs/fig_m2_interactions.png", dpi=120, bbox_inches='tight')
plt.show()

print("\nEfecto principal Tec21 (era_main) en M2:", f"{era_coef_m2:+.4f}")
print("\nCoeficientes de interacción era×variable:")
for lbl, val in zip(labels_inter, inter_vals):
    direction = "AUMENTA riesgo en Tec21" if val > 0 else "REDUCE riesgo en Tec21"
    print(f"  {lbl:35s}: {val:+.4f}  → {direction}")

> **Preguntas de interpretación — Coeficientes de Interacción M2**
>
> 1. **Panel izquierdo**: ¿Qué variables cambiaron de signo o de magnitud importante al pasar de M1 a M2?  
>    *Escribe aquí tu interpretación.*
>
> 2. **Panel derecho**: ¿Cuáles interacciones era×variable tienen efecto protector (negativo) y cuáles amplifican el riesgo (positivo) en el contexto Tec21?  
>    *Escribe aquí tu interpretación.*
>
> 3. ¿El efecto principal de `era_main` es coherente con los efectos de interacción observados?  
>    *Escribe aquí tu interpretación.*

### 3.5.2  Heterogeneidad entre Escuelas: Efectos Aleatorios de M3 (alpha_school y beta_era)

In [ ]:
alpha_school = m3_post["alpha_school"]  # (n_schools,)  riesgo base por escuela
delta_school = m3_post["beta_era"]      # (n_schools,)  impacto Tec21 por escuela

n_schools = len(school_labels)

# ── Figura: 3 paneles ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("M3: Heterogeneidad Estructural entre Escuelas (Efectos Aleatorios ADVI)", fontsize=13, fontweight='bold')

# Panel 1 — riesgo base (alpha_school) ────────────────────────────────────────
order_alpha = np.argsort(alpha_school)
y_pos = np.arange(n_schools)
ax1 = axes[0]
bars_a = ax1.barh(y_pos, alpha_school[order_alpha], color='steelblue', alpha=0.85)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([school_labels[i] for i in order_alpha], fontsize=8)
ax1.axvline(0, color='black', lw=0.8, ls='--')
ax1.set_xlabel('alpha_school (log-odds)'); ax1.set_title('Riesgo Base de Deserción por Escuela')
for bar, val in zip(bars_a, alpha_school[order_alpha]):
    ax1.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
             f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=7.5)

# Panel 2 — impacto Tec21 (delta_school) ─────────────────────────────────────
order_delta = np.argsort(delta_school)
ax2 = axes[1]
colors_d = ['#d62728' if v > 0 else '#1f77b4' for v in delta_school[order_delta]]
bars_d = ax2.barh(y_pos, delta_school[order_delta], color=colors_d, alpha=0.85)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([school_labels[i] for i in order_delta], fontsize=8)
ax2.axvline(0, color='black', lw=0.8, ls='--')
ax2.set_xlabel('beta_era / delta (log-odds)'); ax2.set_title('Impacto Diferencial de Tec21 por Escuela')
for bar, val in zip(bars_d, delta_school[order_delta]):
    ax2.text(val + (0.003 if val >= 0 else -0.003), bar.get_y() + bar.get_height()/2,
             f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=7.5)

# Panel 3 — scatter alpha vs delta ────────────────────────────────────────────
ax3 = axes[2]
sc = ax3.scatter(alpha_school, delta_school, s=120, c=delta_school,
                 cmap='RdYlBu_r', edgecolors='k', linewidths=0.7, zorder=3)
for i, name in enumerate(school_labels):
    ax3.annotate(name, (alpha_school[i], delta_school[i]),
                 textcoords="offset points", xytext=(5, 4), fontsize=7.5)
ax3.axhline(0, color='grey', lw=0.8, ls='--'); ax3.axvline(0, color='grey', lw=0.8, ls='--')
ax3.set_xlabel('alpha_school (riesgo base)'); ax3.set_ylabel('beta_era (impacto Tec21)')
ax3.set_title('Riesgo Base vs Impacto Tec21 por Escuela')
fig.colorbar(sc, ax=ax3, label='beta_era')

fig.tight_layout()
plt.savefig("../outputs/fig_m3_school_effects.png", dpi=120, bbox_inches='tight')
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────────────────
df_schools = pd.DataFrame({
    'Escuela':             school_labels,
    'alpha_school':        alpha_school.round(4),
    'Prob. base (logit)':  [f'{1/(1+np.exp(-a)):.3f}' for a in alpha_school],
    'beta_era (Δ Tec21)':  delta_school.round(4),
    'Dirección':           ['↑ Mayor riesgo' if d > 0 else '↓ Menor riesgo' for d in delta_school],
}).sort_values('beta_era (Δ Tec21)', ascending=False).reset_index(drop=True)

print("\nEfectos aleatorios por escuela (ordenados por impacto Tec21):")
print(df_schools.to_string(index=False))

> **Preguntas de interpretación — Heterogeneidad entre Escuelas (M3)**
>
> 1. **Panel 1 (alpha_school)**: ¿Qué escuela tiene el mayor riesgo base de deserción y cuál el menor?  
>    ¿La diferencia es grande en términos de probabilidad?  
>    *Escribe aquí tu interpretación.*
>
> 2. **Panel 2 (beta_era / delta_school)**: ¿En qué escuelas la transición a Tec21 *aumentó* el riesgo de deserción y en cuáles lo *redujo*?  
>    *Escribe aquí tu interpretación.*
>
> 3. **Panel 3 (scatter)**: ¿Existe correlación entre el riesgo base de la escuela y el impacto de Tec21?  
>    ¿Las escuelas con mayor riesgo previo experimentaron un cambio mayor o menor bajo Tec21?  
>    *Escribe aquí tu interpretación.*
>
> 4. **Síntesis**: Con base en §3.5.1 y §3.5.2, responde en 3-5 oraciones la pregunta de investigación:  
>    *¿De qué manera la transición al modelo Tec21 alteró la heterogeneidad estructural de la deserción entre escuelas, y qué variables han ganado peso predictivo en este nuevo régimen?*  
>    *Escribe aquí tu respuesta integrada.*

---
## 3.6  SHAP Values — M3 Multilevel ADVI (Pre-Tec21 vs Tec21 por Escuela)

M3 es un modelo log-lineal: log-odds = α_escuela[s] + β_era[s]·era + X·β.
Los valores SHAP se calculan analíticamente aprovechando la aditividad lineal exacta:

- **Contribución de característica j**: (x_j − E[x_j]) · β_j  
- **Contribución de escuela**: α_escuela[s] − E[α_escuela]  
- **Contribución de era**: β_era[s]·era − E[β_era·era]

Las dos figuras siguientes descomponen estas contribuciones separando los datos de prueba en era **Pre-Tec21** y **Tec21**, coloreando cada punto por escuela.

In [ ]:
# ── Compute M3 SHAP values (linear exact decomposition) ─────────────────────
beta_m3     = m3_post["beta"]           # (62,)
alpha_s_m3  = m3_post["alpha_school"]  # (6,)
beta_era_m3 = m3_post["beta_era"]      # (6,)

# Background reference: test set feature means (X_train_m3 not stored in artifacts)
bg_m3 = X_test_m3.mean(axis=0)

# Feature SHAP: (x_i - E[x]) * beta
shap_feat_m3 = (X_test_m3 - bg_m3) * beta_m3  # (n_test, 62)

# School intercept SHAP: deviation from mean school intercept
shap_school_m3 = alpha_s_m3[school_test] - alpha_s_m3[school_test].mean()

# Era effect SHAP: deviation from mean era contribution
era_contrib   = beta_era_m3[school_test] * era_test
shap_era_m3   = era_contrib - era_contrib.mean()

# Combined SHAP matrix: features + school intercept + era effect
shap_all_m3  = np.column_stack([shap_feat_m3, shap_school_m3, shap_era_m3])
feat_ext_m3  = list(feat_m3) + ["school_intercept", "era_effect"]

# Base value: mean log-odds prediction across test set
base_val_m3 = float(bg_m3 @ beta_m3 + alpha_s_m3[school_test].mean() + era_contrib.mean())

# Verify additivity: sum(SHAP_i) must equal f(x_i) - base_val
pred_logodds = alpha_s_m3[school_test] + beta_era_m3[school_test] * era_test + X_test_m3 @ beta_m3
residual     = pred_logodds - base_val_m3 - shap_all_m3.sum(axis=1)
print(f"Max SHAP residual : {np.abs(residual).max():.2e}  (debe ser ~0)")
print(f"Base log-odds     : {base_val_m3:.4f}  →  P ≈ {expit(base_val_m3):.4f}")
print(f"Matriz SHAP       : {shap_all_m3.shape}  ({len(feat_ext_m3)} columnas incl. escuela y era)")

In [ ]:
# ── SHAP por escuela: Pre-Tec21 vs Tec21 (2×3 grid, una gráfica por escuela) ─
N_FEAT = 12  # características a mostrar por escuela

fig, axes = plt.subplots(2, 3, figsize=(20, 14), constrained_layout=True)
axes = axes.ravel()

for sid, (ax, school_name) in enumerate(zip(axes, school_labels)):
    s_mask_pre = (school_test == sid) & (era_test == 0)
    s_mask_tec = (school_test == sid) & (era_test == 1)
    n_pre, n_tec = s_mask_pre.sum(), s_mask_tec.sum()

    shap_pre = shap_all_m3[s_mask_pre].mean(axis=0) if n_pre > 0 else np.zeros(len(feat_ext_m3))
    shap_tec = shap_all_m3[s_mask_tec].mean(axis=0) if n_tec > 0 else np.zeros(len(feat_ext_m3))

    # Select top N features by the larger of the two era means (absolute value)
    top_idx_s   = np.argsort(np.maximum(np.abs(shap_pre), np.abs(shap_tec)))[::-1][:N_FEAT]
    feat_labels = [feat_ext_m3[i] for i in top_idx_s][::-1]  # reversed: top at top of chart
    pre_vals    = shap_pre[top_idx_s][::-1]
    tec_vals    = shap_tec[top_idx_s][::-1]

    y      = np.arange(N_FEAT)
    height = 0.38

    ax.barh(y + height / 2, pre_vals, height,
            label=f"Pre-Tec21 (n={n_pre:,})", color="#1f77b4", alpha=0.85)
    ax.barh(y - height / 2, tec_vals, height,
            label=f"Tec21     (n={n_tec:,})", color="#ff7f0e", alpha=0.85)

    ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels(feat_labels, fontsize=7.5)
    ax.set_title(school_name, fontsize=10, fontweight="bold")
    ax.set_xlabel("SHAP promedio (log-odds)", fontsize=8)
    ax.legend(fontsize=7.5, loc="lower right")
    ax.grid(axis="x", alpha=0.25)

fig.suptitle(
    "M3 – Multilevel ADVI: Efecto de Cada Variable por Escuela antes y después de Tec21\n"
    "(barras azules = Pre-Tec21, naranja = Tec21; top 12 características por escuela)",
    fontsize=12, fontweight="bold",
)
plt.savefig("fig_shap_m3_by_era_school.png", dpi=150, bbox_inches="tight")
plt.show()
